# Figures 3, 4 and 7 - Centre-of-mass static structure factor S(q)

Many-body Monte Carlo simulations with **Faunus**: N = 20 rigid antibodies at
amino-acid resolution in a cubic box, **hard-sphere** interactions only. For
each run the centre-of-mass scattering is accumulated (`debye_COM.dat`) and
gives the centre-of-mass static structure factor S(q).

Runs cover `mAb-1` (7 and 57 mM structures) at three protein concentrations
(50, 100, 150 mg/mL), three independent replicas each. All runs live under an
`I_q/` sub-directory, one folder per structure/concentration/replica.

Plots:
* **Figure 4** - S(q) per concentration (mAb-1, 7 mM structure)
* **Figure 3** - S(0) vs concentration (both structures), extrapolated from S(q)
* **Figure 7** - S(q) grid, both structures x three concentrations

**How to run:** install the environment, then run top to bottom. Edit
`run_faunus()` to submit to a cluster instead. With `micro = 10` these are short
runs; increase `micro` for production-quality statistics.

## Import packages

In [ ]:
import os
import subprocess
import numpy as np
from scipy import interpolate
from jinja2 import Template
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 16, "legend.frameon": True, "figure.figsize": [5, 5],
                     "xtick.major.size": 7, "ytick.major.size": 7, "legend.labelspacing": 1})


## Functions

In [ ]:
def Length_in_gl(cp, N_of_proteins, Mw):
    """Cubic box side (angstrom) giving protein concentration cp (mg/mL)
    for N_of_proteins molecules of molar mass Mw (g/mol)."""
    return np.cbrt(N_of_proteins / (cp * (1 / Mw) * 6.022e23 * 1e-27))


def create_input(workdir, structure, macro, micro, nstep, nskip, L, N):
    """Render the Faunus input (input.yml). L is the cubic box side (angstrom),
    N the number of antibodies inserted."""
    with open("input.yml", "w") as f:
        return f.write(Input.render(workdir=workdir, structure=structure,
                                    macro=macro, micro=micro, nstep=nstep,
                                    nskip=nskip, L=L, N=N))


def run_faunus(use_state=False):
    """Run Faunus locally in the current run directory.

    Requires `faunus` and `yason.py` on PATH (installed by the conda
    environment, see ../environment.yml). Output goes to `out`, errors to `err`.
    To run on a cluster instead, replace the body of this function with your
    own submission command (e.g. write an sbatch script and call `sbatch`)."""
    state = "--state state.json " if use_state else ""
    cmd = ("set -o pipefail; export OMP_NUM_THREADS=1; "
           f"yason.py input.yml | faunus {state}-v 6 > out 2> err")
    subprocess.run(["bash", "-c", cmd], check=True)


# Many-body runs - hard-sphere potential

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------

Input = Template("""
temperature: 298.15
geometry: {type: cuboid, length: [{{L}},{{L}},{{L}}]}
mcloop: {micro: {{micro}}, macro: {{macro}}}
random: {seed: hardware}
energy:
    - nonbonded:
        cutoff_g2g: 180
        default:
            - hardsphere:
                mixing: arithmetic 

atomlist:
    - CTR:  {mw: 16.0,  sigma: 3.13, eps: 0, q: 0 }
    - HCTR: {mw: 16.0,  sigma: 3.13, eps: 0, q: 0 }
    - NTR:  {mw: 14.0,  sigma: 2.99, eps: 0, q: 0 }
    - HNTR: {mw: 14.0,  sigma: 2.99, eps: 0, q: 0 }
    - ALA:  {mw: 66.0,  sigma: 5.02, eps: 0, q: 0 }
    - ILE:  {mw: 102.0, sigma: 5.8,  eps: 0, q: 0 }
    - LEU:  {mw: 102.0, sigma: 5.8,  eps: 0, q: 0 }
    - MET:  {mw: 122.0, sigma: 6.16, eps: 0, q: 0 }
    - PHE:  {mw: 138.0, sigma: 6.41, eps: 0, q: 0 }
    - VAL:  {mw: 90.0,  sigma: 5.56, eps: 0, q: 0 }
    - PRO:  {mw: 90.0,  sigma: 5.56, eps: 0, q: 0 }
    - TRP:  {mw: 176.0, sigma: 6.95, eps: 0, q: 0 }
    - GLY:  {mw: 54.0,  sigma: 4.69, eps: 0, q: 0 }
    - ARG:  {mw: 144.0, sigma: 6.50, eps: 0, q: 0 }
    - HARG: {mw: 144.0, sigma: 6.50, eps: 0, q: 0 }
    - LYS:  {mw: 116.0, sigma: 6.05, eps: 0, q: 0 }
    - HLYS: {mw: 116.0, sigma: 6.05, eps: 0, q: 0 }
    - ASP:  {mw: 110.0, sigma: 5.94, eps: 0, q: 0 }
    - HASP: {mw: 110.0, sigma: 5.94, eps: 0, q: 0 }
    - LASP: {mw: 110.0, sigma: 5.94, eps: 0, q: 0 }
    - GLU:  {mw: 122.0, sigma: 6.16, eps: 0, q: 0 }
    - HGLU: {mw: 122.0, sigma: 6.16, eps: 0, q: 0 }
    - LGLU: {mw: 122.0, sigma: 6.16, eps: 0, q: 0 }
    - GLN:  {mw: 120.0, sigma: 6.12, eps: 0, q: 0 }
    - ASN:  {mw: 108.0, sigma: 5.91, eps: 0, q: 0 }
    - HIS:  {mw: 130.0, sigma: 6.29, eps: 0, q: 0 }
    - HHIS: {mw: 130.0, sigma: 6.29, eps: 0, q: 0 }
    - SER:  {mw: 82.0,  sigma: 5.39, eps: 0, q: 0 }
    - THR:  {mw: 94.0,  sigma: 5.64, eps: 0, q: 0 }
    - TYR:  {mw: 154.0, sigma: 6.65, eps: 0, q: 0 }
    - HTYR: {mw: 154.0, sigma: 6.65, eps: 0, q: 0 }
    - CYS:  {mw: 103.0, sigma: 5.72, eps: 0, q: 0 }
    - HCYS: {mw: 103.0, sigma: 5.72, eps: 0, q: 0 }
    - CYX:  {mw: 103.0, sigma: 5.72, eps: 0, q: 0 }              

moleculelist:
    - mAb_aa:
        structure: {{workdir}}/../pqr_inputs/{{structure}}
        keepcharges: False
        keeppos: False
        rigid: True

insertmolecules:
    - mAb_aa: {N: {{N}}, inactive: False}
moves:    
    - moltransrot: {molecule: mAb_aa, dir: [1,1,1], dprot: 0.2, dp: 100 }

analysis:
    - sanity: {nstep: {{nstep}} }
    - molecule_density: {nstep: {{nstep}}}
    - savestate: {file: state.json}
    - savestate: {file: confout.pqr}
    - xtcfile: {file: traj.xtc, nstep: {{nstep}}, nskip: {{nskip}} }
    - scatter: {file: debye_COM.dat, nstep: 1, molecules: ["mAb_aa"], com: True, scheme: explicit, pmax: 15 } 

""")

Mw = 148000                              # antibody molar mass, g/mol
cp_range = [50, 100, 150]                # protein concentrations, mg/mL
N = 20                                   # antibodies per box
structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr']
replicas = [0, 1, 2]                     # three independent replicas

micro = 10
macro = 1000
nstep = 1000
nskip = 0

os.makedirs('I_q', exist_ok=True)
os.chdir('I_q')
iq_dir = os.getcwd()

for structure in structures_list:
    name = os.path.splitext(structure)[0]
    for cp in cp_range:
        L = Length_in_gl(cp, N, Mw)      # box side for this concentration
        for r in replicas:
            rundir = f"{name}_cp{cp}_r{r}"
            os.makedirs(rundir, exist_ok=True)
            os.chdir(rundir)
            create_input(workdir, structure, macro, micro, nstep, nskip, L, N)
            run_faunus(use_state=os.path.isfile('state.json'))
            os.chdir(iq_dir)
os.chdir(workdir)


# Plot $S^{cm}(q)$ - mAb-1 (7 mM) - Figure 4

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------


### Create Figure: S_com(q), one panel per concentration ###
plt.rcParams.update({'font.size':16,'legend.frameon':True,'figure.figsize':[6,12],
                     'xtick.major.size':7,'ytick.major.size':7,'legend.labelspacing':1})

cp_range = [50, 100, 150]             # one panel per concentration
panel_labels = ['A', 'B', 'C']
replicas = [0, 1, 2]                  # must match the run cell
q_extrap = 0.07                       # extrapolate down to this q [nm^-1]
#######################################################################################################################################################


structures_list = ['mAb-1_7mM.pqr']

fig, ax = plt.subplots(nrows=len(cp_range), ncols=1, sharex=True, sharey=True)
if len(cp_range) == 1:
    ax = [ax]                          # make it indexable when there's one panel

S_com_extrap = {}                      # S_com(0.07) per concentration, if needed later

for k, cp in enumerate(cp_range):
    for structure in structures_list:
        name = os.path.splitext(structure)[0]

        ### Load all replicas ###
        S_reps = []
        for r in replicas:
            I_dir = os.path.join(workdir, 'I_q', f"{name}_cp{cp}_r{r}")
            q, S_r = np.loadtxt(os.path.join(I_dir, 'debye_COM.dat'),
                                unpack=True, skiprows=1, usecols=(0,1))
            S_reps.append(S_r)

        # q from Faunus is in 1/Å -> convert to 1/nm for the plot
        q_nm = q * 10

        ### Replica average ###
        S_com = np.mean(S_reps, axis=0)

        ### Single replica points on top (faint) ###
        for r, S_r in zip(replicas, S_reps):
            ax[k].plot(q_nm, S_r, 'o',
                       color='black',
                       markerfacecolor='none',
                       markeredgecolor='black',
                       markersize=4,
                       alpha=0.25)

        ### Average line ###
        ax[k].plot(q_nm, S_com, 'o-',
                   color='black',
                   markerfacecolor='none',
                   markeredgecolor='black',
                   markersize=6,
                   label=f'$S_{{com}}$(q), $c_p$ = {cp} mg/mL')

        ### Extrapolation down to q = 0.07 nm^-1 ###
        S_inter = interpolate.interp1d(q_nm, S_com, fill_value='extrapolate')
        q_low = np.arange(q_extrap, q_nm.min(), 0.001)

        ax[k].plot(q_low, S_inter(q_low), '--',
                   color='black',
                   alpha=0.8)


        S_com_extrap[cp] = S_inter(q_extrap)
        # print(f"cp = {cp}: S_com({q_extrap}) = {S_inter(q_extrap):.4f}")

        # log x scale, 0.1 and 1 as reference values
        ax[k].set_xscale('log')
        ax[k].set_xlim(0.06, 3.0)
        ax[k].set_ylim(0, 1.5)
        ax[k].set_xticks([0.1, 1])
        ax[k].get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax[k].minorticks_off()

        ax[k].set_ylabel("S(q)", fontsize=25)
        ax[k].set_title(panel_labels[k] + f'  ($c_p$ = {cp} mg/mL)',
                        loc='left', fontsize=12)


# x label only under the last panel
ax[-1].set_xlabel("q [nm$^{-1}$]", fontsize=25)

# legend in the bottom panel, like the reference figure
ax[-1].legend(fontsize=10, frameon=True, loc='lower right')

plt.tight_layout()
plt.show()


# Plot $S(0)$ vs concentration - Figure 3

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------


cp_range = [50, 100, 150]
replicas = [0, 1, 2]                   # must match the run cell
q_extrap = 0.001                       # extrapolate down to this q [nm^-1]
#######################################################################################################################################################


structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr' ]

# color and label per structure
style = {
    'mAb-1_7mM.pqr':  ('black', 'mAb-1, I = 7 mM'),
    'mAb-1_57mM.pqr': ('red',  'mAb-1, I = 57 mM'),
}

plt.rcParams.update({'font.size':16,'legend.frameon':True,
                     'figure.figsize':[7,5],
                     'xtick.major.size':7,'ytick.major.size':7})
fig, ax = plt.subplots(nrows=1, ncols=1)

for structure in structures_list:
    name = os.path.splitext(structure)[0]
    color, label = style[structure]

    S0_avg, S0_reps, S0_std = [], [], []

    for cp in cp_range:
        # extrapolated S(0) per replica
        S0_r = []
        for r in replicas:
            I_dir = os.path.join(workdir, 'I_q', f"{name}_cp{cp}_r{r}")
            q, S_r = np.loadtxt(os.path.join(I_dir, 'debye_COM.dat'),
                                unpack=True, skiprows=1, usecols=(0,1))
            q_nm = q * 10
            S_inter = interpolate.interp1d(q_nm, S_r,
                                            fill_value='extrapolate')
            S0_r.append(S_inter(q_extrap))

        S0_reps.append(S0_r)
        S0_avg.append(np.mean(S0_r))
        S0_std.append(np.std(S0_r))

    S0_avg = np.array(S0_avg)
    S0_std = np.array(S0_std)

    # single replica points (on top, faint)
    for k, cp in enumerate(cp_range):
        ax.plot(cp * np.ones(len(replicas)), S0_reps[k], 'o',
                color=color, markerfacecolor='none',
                markeredgecolor=color, markersize=5, alpha=0.3)

    # average with error bar (std over replicas)
    ax.errorbar(cp_range, S0_avg, yerr=S0_std, fmt='o',
                color=color, markerfacecolor='none',
                markeredgecolor=color, markersize=8,
                capsize=5, linewidth=1.5, label=label)

ax.set_xlabel("$c_p$ [mg/mL]", fontsize=25)
ax.set_ylabel("S(0)", fontsize=25)
ax.legend(fontsize=10, frameon=True, loc='lower left')

plt.tight_layout()
plt.show()


# Plot $S^{cm}(q)$ grid - Figure 7

In [ ]:
# --- Define the working directory ONCE ---
try:
    os.chdir(workdir)
except (NameError, FileNotFoundError):
    workdir = os.getcwd()

workdir = os.getcwd()
#-------------------------------------------


### Create Figure: S_com(q), rows = structures, columns = concentrations ###
plt.rcParams.update({'font.size':16,'legend.frameon':True,'figure.figsize':[14,9],
                     'xtick.major.size':7,'ytick.major.size':7,'legend.labelspacing':1})

structures_list = ['mAb-1_7mM.pqr', 'mAb-1_57mM.pqr']   # one row per mAb
cp_range = [50, 100, 150]                                # one column per concentration
replicas = [0, 1, 2]                  # must match the run cell
q_extrap = 0.07                       # extrapolate down to this q [nm^-1]
panel_labels = [['A', 'B', 'C'],
                ['D', 'E', 'F']]
row_titles = {'mAb-1_7mM.pqr':  'mAb-1, I = 7 mM',
              'mAb-1_57mM.pqr': 'mAb-1, I = 57 mM'}
#######################################################################################################################################################


fig, ax = plt.subplots(nrows=len(structures_list), ncols=len(cp_range),
                       sharex=True, sharey=True)

S_com_extrap = {}                      # (structure, cp) -> S_com(q_extrap)

for i, structure in enumerate(structures_list):
    name = os.path.splitext(structure)[0]

    for j, cp in enumerate(cp_range):

        ### Load all replicas ###
        S_reps = []
        for r in replicas:
            I_dir = os.path.join(workdir, 'I_q', f"{name}_cp{cp}_r{r}")
            q, S_r = np.loadtxt(os.path.join(I_dir, 'debye_COM.dat'),
                                unpack=True, skiprows=1, usecols=(0,1))
            S_reps.append(S_r)

        # q from Faunus is in 1/Å -> convert to 1/nm
        q_nm = q * 10

        ### Replica average ###
        S_com = np.mean(S_reps, axis=0)

        ### Single replica points (faint) ###
        for r, S_r in zip(replicas, S_reps):
            ax[i,j].plot(q_nm, S_r, 'o',
                         color='black', markerfacecolor='none',
                         markeredgecolor='black', markersize=4, alpha=0.25)

        ### Average line ###
        ax[i,j].plot(q_nm, S_com, 'o-',
                     color='black', markerfacecolor='none',
                     markeredgecolor='black', markersize=6,
                     label=f'$c_p$ = {cp} mg/mL')

        ### Extrapolation down to q = 0.07 nm^-1 ###
        S_inter = interpolate.interp1d(q_nm, S_com, fill_value='extrapolate')
        q_low = np.arange(q_extrap, q_nm.min(), 0.001)
        ax[i,j].plot(q_low, S_inter(q_low), '--', color='black', alpha=0.8)

        S_com_extrap[(structure, cp)] = S_inter(q_extrap)

        ### Axes ###
        ax[i,j].set_xscale('log')
        ax[i,j].set_xlim(0.06, 3.0)
        ax[i,j].set_ylim(0, 1.5)
        ax[i,j].set_xticks([0.1, 1])
        ax[i,j].get_xaxis().set_major_formatter(plt.ScalarFormatter())
        ax[i,j].minorticks_off()

        ax[i,j].set_ylabel("S(q)", fontsize=25)
        ax[i,j].set_title(panel_labels[i][j] + f'  ({row_titles[structure]}, $c_p$ = {cp} mg/mL)',
                          loc='left', fontsize=12)

        np.savetxt(os.path.join(workdir, f'S_com_{name}_{cp}mgmL.txt'),
                   np.c_[q_nm, S_com])

# x label only under the bottom row
for j in range(len(cp_range)):
    ax[-1,j].set_xlabel("q [nm$^{-1}$]", fontsize=25)


plt.tight_layout()
plt.show()
